# Bayesian Hyperparameter Space Optimization and Automated Model Auditing Pipeline

In [21]:
%load_ext autoreload
%autoreload 2

import os
import mlflow
import optuna
from optuna.integration.mlflow import MLflowCallback
from optuna.pruners import MedianPruner
import pandas as pd

from src import optuna_optimization as utils

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1. Path Configuration & MLflow Backend Binding

In [22]:
RANDOM_STATE = 42
N_SPLITS = 5
EXPERIMENT_NAME = "customer-churn-optuna"

ROOT_DIR = os.path.abspath("../")
DB_PATH = os.path.join(ROOT_DIR, "mlflow.db")
ARTIFACTS_DIR = os.path.join(ROOT_DIR, "mlartifacts")

# Set the tracking URI immediately to lock it to SQLite
mlflow.set_tracking_uri(f"sqlite:///{DB_PATH}")

# Explicitly create the experiment with your designated mlartifacts folder path
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
if experiment is None:
    mlflow.create_experiment(
        name=EXPERIMENT_NAME,
        artifact_location=f"file://{ARTIFACTS_DIR}"
    )

# Active the experiment scope
mlflow.set_experiment(EXPERIMENT_NAME)

<Experiment: artifact_location='file:///Users/hector.vargas/repos/ml_hands_on_project/mlartifacts', creation_time=1780446392364, experiment_id='4', last_update_time=1780446392364, lifecycle_stage='active', name='customer-churn-optuna', tags={}, trace_location=None, workspace='default'>

## 2. Custom Source Code Ingestion

In [23]:
X_train = pd.read_csv("../data/processed/raw_features/X_train.csv")
X_test = pd.read_csv("../data/processed/raw_features/X_test.csv")
y_train = pd.read_csv("../data/processed/target/y_train.csv").squeeze()
y_test = pd.read_csv("../data/processed/target/y_test.csv").squeeze()

In [24]:
# Mapping Runtime Feature Schemas
nomod_columns = ['HasCrCard', 'IsActiveMember']
dummyfy_columns = ['Card Type', 'NumOfProducts', 'Geography', 'Gender']
norm_std_columns = ['Balance', 'Point Earned', 'CreditScore', 'Age', 'Tenure', 'Satisfaction Score', 'EstimatedSalary']

## 2. Orchestrate and Initialize Search Parameter Studies

In [25]:
# Initialize Integrated MLflow Tracking Integration Callbacks
mlflow_callback = MLflowCallback(
    tracking_uri=mlflow.get_tracking_uri(),
    metric_name="average_precision",
    create_experiment=False, # Keeps Optuna from trying to override our custom artifact path
    mlflow_kwargs={"nested": True}
)

study = optuna.create_study(
    direction='maximize',
    pruner=MedianPruner(n_startup_trials=5, n_warmup_steps=0)
)

/var/folders/bv/50x24wc545x5mclk_t88ryrc0000gn/T/ipykernel_11902/1472475305.py:2: ExperimentalWarning: MLflowCallback is experimental (supported from v1.4.0). The interface can change in the future.
  mlflow_callback = MLflowCallback(
[I 2026-06-02 18:26:32,427] A new study created in memory with name: no-name-c7565cb9-47bc-438e-b931-85a5858ce0ba


In [26]:
# Instantiate the cross-validation objective callable
objective_function = utils.ObjectiveCV(
    X=X_train, y=y_train, 
    cat_cols=dummyfy_columns, num_cols=norm_std_columns, pass_cols=nomod_columns,
    n_splits=N_SPLITS, random_state=RANDOM_STATE
)

## 3. Run Optimization Search Workspace Execution Window

In [27]:
with mlflow.start_run(run_name="optuna_search_parent"):
    
    study.optimize(
        objective_function,
        n_trials=50,
        callbacks=[mlflow_callback]
    )

    # Log global best performance summary attributes
    mlflow.log_metric("best_auc", study.best_value)
    mlflow.log_params(study.best_params)

    # Reconstruct and serialize top performing candidate artifacts pipeline
    best_pipeline = utils.build_pipeline(
        study.best_trial, 
        cat_cols=dummyfy_columns, num_cols=norm_std_columns, pass_cols=nomod_columns,
        random_state=RANDOM_STATE
    )

    # This internally calls artifact logging functions; they will now map straight to mlartifacts/
    utils.evaluate_and_log_best_model(
        best_pipeline=best_pipeline,
        X_train=X_train, y_train=y_train,
        X_test=X_test, y_test=y_test,
        cat_cols=dummyfy_columns, num_cols=norm_std_columns, pass_cols=nomod_columns
    )

[I 2026-06-02 18:26:34,711] Trial 0 finished with value: 0.6692671966929613 and parameters: {'scaler': 'std', 'encoder': 'drop_first', 'model': 'rf', 'rf_n_estimators': 205, 'rf_max_depth': 5, 'rf_min_samples_split': 12, 'rf_min_samples_leaf': 7}. Best is trial 0 with value: 0.6692671966929613.
[I 2026-06-02 18:26:36,972] Trial 1 finished with value: 0.6801945779053459 and parameters: {'scaler': 'minmax', 'encoder': 'no_drop', 'model': 'rf', 'rf_n_estimators': 458, 'rf_max_depth': 9, 'rf_min_samples_split': 5, 'rf_min_samples_leaf': 3}. Best is trial 1 with value: 0.6801945779053459.
[I 2026-06-02 18:26:38,698] Trial 2 finished with value: 0.6730275800532535 and parameters: {'scaler': 'std', 'encoder': 'drop_first', 'model': 'rf', 'rf_n_estimators': 185, 'rf_max_depth': 20, 'rf_min_samples_split': 11, 'rf_min_samples_leaf': 10}. Best is trial 1 with value: 0.6801945779053459.
[I 2026-06-02 18:26:38,848] Trial 3 finished with value: 0.6945572085828193 and parameters: {'scaler': 'std', '

## 4. Display Session Optimization Diagnostics Results

In [28]:
print(f"\nTop Optimization Average Precision Score Target: {study.best_value:.4f}")
print("\nOptimal Parameter Combinations Selected:")
for parameter_key, parameter_value in study.best_params.items():
    print(f" * {parameter_key}: {parameter_value}")


Top Optimization Average Precision Score Target: 0.7063

Optimal Parameter Combinations Selected:
 * scaler: robust
 * encoder: drop_first
 * model: xgb
 * xgb_n_estimators: 968
 * xgb_max_depth: 5
 * xgb_learning_rate: 0.06832797329643628
 * xgb_subsample: 0.9354669812000643
 * xgb_colsample_bytree: 0.6604095125174853
 * xgb_min_child_weight: 2
 * xgb_gamma: 5.060087192357632
 * xgb_reg_alpha: 0.00026126473493891814
 * xgb_reg_lambda: 2.0727554085865994e-06
 * xgb_scale_pos_weight: 1.981967652946261


Best average_precision: **0.7077109199922507**

Optimal Parameter Combinations Selected:
 * scaler: robust
 * encoder: drop_first
 * model: xgb
 * xgb_n_estimators: 588
 * xgb_max_depth: 5
 * xgb_learning_rate: 0.06090819562411312
 * xgb_subsample: 0.967454071450258
 * xgb_colsample_bytree: 0.6262720546196823
 * xgb_min_child_weight: 3
 * xgb_gamma: 3.2784302986561005
 * xgb_reg_alpha: 0.0006480121274906255
 * xgb_reg_lambda: 7.441405323298544e-05
 * xgb_scale_pos_weight: 1.8090219464356987


Top Optimization Average Precision Score Target: **0.7079**

Optimal Parameter Combinations Selected:
 * scaler: minmax
 * encoder: drop_first
 * model: xgb
 * xgb_n_estimators: 967
 * xgb_max_depth: 5
 * xgb_learning_rate: 0.05903303186469791
 * xgb_subsample: 0.9204856064383659
 * xgb_colsample_bytree: 0.6118872682736142
 * xgb_min_child_weight: 2
 * xgb_gamma: 4.2451076897824676
 * xgb_reg_alpha: 0.00019575646965291755
 * xgb_reg_lambda: 0.0005189779456101976
 * xgb_scale_pos_weight: 1.6516119336541517